In [6]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.config import SolverConfig
from src.models.spacecraft import Spacecraft
from src.models.body import Body
from src.models.orbit import Orbit_Eq
from src.optimizer import Optimizer
from src.utils import cart2eq
from src.planet_propagator import earth_state, mars_state
from src.qlaw import QlawController

In [7]:
t0 = 0.0

def run_trajectory(opt, t0, tf, y0):
    sol = opt.propagator.forward(
        y0=y0,
        tspan=(t0, tf),
        control=opt.qlaw_control()
    )

    return sol

In [8]:
t0 = 0.0
tf = 25920000.0

Psyche = Spacecraft()
controller = QlawController(Psyche)

r0v0=np.array([
            1.0000e11, 0.0, 0.0,
            0.0, 3.0e4, 0.0,
            1000.0
        ])
m0 = Psyche.m0

y0 = np.hstack((r0v0, m0))

mee_target = [1.0100e11, 0.0, 0.0, 0.0, 0.0]
u,q = controller.control(mu=1.327e20, y=y0, target=mee_target, Qprev=1.0)

print("u =", u)
print("Q =", q)
print("tangential sign =", np.sign(u[1]))

u = [-1.12000000e+00 -1.30294215e-17  0.00000000e+00]
Q = 60505117528103.32
tangential sign = -1.0


In [9]:
t0 = 0.0
tf = 25920000.0

Psyche = Spacecraft()
controller = QlawController(Psyche)

r0v0=np.array([
            1.0000e11, 0.0, 0.0,
            0.0, 3.0e4, 0.0,
            1000.0
        ])
m0 = Psyche.m0

y0 = np.hstack((r0v0, m0))

mee_target = [9.9000e10, 0.0, 0.0, 0.0, 0.0]
u,q = controller.control(mu=1.327e20, y=y0, target=mee_target, Qprev=1.0)

print("u =", u)
print("Q =", q)
print("tangential sign =", np.sign(u[1]))

u = [-1.12000000e+00 -1.67765675e-17  0.00000000e+00]
Q = 57453389591382.74
tangential sign = -1.0


In [10]:
t0 = 0.0
tf = 25920000.0

Psyche = Spacecraft()
controller = QlawController(Psyche)

r0v0=np.array([
            1.0000e11, 0.0, 0.0,
            0.0, 3.0e4, 0.0,
            1000.0
        ])
m0 = Psyche.m0

y0 = np.hstack((r0v0, m0))

mee_target = [1.0100e11, 0.0, 0.0, 1e-4, 0.0]
u,q = controller.control(mu=1.327e20, y=y0, target=mee_target, Qprev=1.0)

print("u =", u)
print("Q =", q)
print("normal component =", u[2])

u = [-1.11999776e+00 -1.30294131e-17  2.23782925e-03]
Q = 60505226531646.41
normal component = 0.0022378292499829187


In [ ]:
t0 = 0.0

def run_trajectory(opt, t0, tf, y0, dt=86400.0):
    t = t0
    y = y0
    ts = [t0]
    ys = [y0]

    while t < t0+tf:
        t_next = min(t + dt, t0+tf)
        sol = opt.propagator.forward(
            y0=y,
            tspan=(t, t_next),
            control=opt.qlaw_control()
        )
        y = sol.y[:, -1]
        t = t_next
        ts.extend(sol.t[1:])
        ys.extend(sol.y.T[1:])

    return np.array(ts), np.array(ys).T



Earth = Body(name="earth", mu=3.986e14)
Mars = Body(name="mars", mu=4.283e13)
Sun = Body(name="sun", mu=1.327e20)
Psyche = Spacecraft()

r_E, v_E = earth_state(t0)

r0 = r_E + np.array([1e8, 0, 0])
v0 = v_E + np.array([0, 3.13e3, 0])
m0 = Psyche.m0
y0 = np.hstack((r0, v0, m0))

r_M, v_M = mars_state(t0+tf)
mee_target = cart2eq(np.hstack((r_M,v_M)))[0:5]

cfg = SolverConfig()
opt = Optimizer(cfg, Psyche, target_orbit=mee_target)

ts, sol = run_trajectory(opt, t0, tf, y0)

yf = sol[:, -1]


print("Best launch epoch:", best_t0/86400)
print("Best TOF [days]:", best_tf / 86400)
print("Best cost:", best_cost)

sol = best_sol
print("Final state:", sol[:, -1])